# 营销触达的增量收益评估

公开数据独立复盘，非本人执行的线上实验。金额为美元、观察窗口为两周。

建议先做 `docs/learning_path.md` 中对应练习，再运行参考代码。本笔记本逐步重新计算关键表，并与运行入口产生的结果核对。

## 1. 环境与分析约定

主指标：全部随机分组客户的人均收入。辅助指标：购买率、访问率。双侧比较两种邮件各自与不发邮件的差异；主指标采用 Holm 校正。这是复盘约定，不是原实验事前注册。

In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if not (ROOT / 'config.json').exists():
    ROOT = ROOT.parent
assert (ROOT / 'config.json').exists(), '请从项目根目录或 notebooks 目录运行'
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
from IPython.display import display, Markdown, Image
from abtest.data import download, validate, prepare, sql_metrics, reconcile, ARMS
from abtest.statistics import *
config = json.loads((ROOT / 'config.json').read_text(encoding='utf-8'))
validate_config(config)
print('Python:', sys.executable)
config

## 2. 质量检查

无真实用户 ID，不能按属性相同直接去重。缺失结果不能填零。`source_row_id` 只是原文件行号。

In [ ]:
raw = pd.read_csv(download(ROOT))
quality = validate(raw)
df = prepare(raw)
display(pd.DataFrame({'dtype': raw.dtypes.astype(str), 'missing': raw.isna().sum()}))
quality

## 3. SQL 汇总与独立核对

先回答：如果先过滤 `visit=1`，改变了什么？不要将实验后筛选得到的人群效果解释为全体客户效果。

In [ ]:
# 表 1：查看 customers 原始表的前 5 行。
preview_sql = 'SELECT * FROM customers LIMIT 5;'
display(sql_metrics(df, preview_sql))

# 表 2：你在 exercises/01_metrics.sql 中写的三组指标汇总。
sql = (ROOT / 'exercises/01_metrics.sql').read_text(encoding='utf-8')
print(sql)
metrics = sql_metrics(df, sql)
reconcile(df, metrics)
display(metrics)

## 4. 分流比例与实验前平衡

SRM 检验阈值是诊断约定。没有异常不证明分流与数据采集全过程正确。SMD 的 0.1 是经验参考线。

In [ ]:
srm, balance = randomization_checks(df, config)
display(srm)
display(balance.sort_values('abs_smd', ascending=False).head(12))

## 5. 收入效果与稳健性

Welch 允许方差不同；Bootstrap 在两组内分别重抽客户，包含零消费。报告绝对增量与区间，再看相对提升。

In [ ]:
effects = experiment_effects(df, config)
display(effects[effects.metric == 'spend'])
# 用 scipy 独立核对 Welch 的 p 值。
from scipy.stats import ttest_ind
for arm in ARMS[1:]:
    reference = ttest_ind(df.loc[df.segment == arm, 'spend'], df.loc[df.segment == ARMS[0], 'spend'], equal_var=False).pvalue
    actual = effects.loc[(effects.arm == arm) & (effects.metric == 'spend'), 'p_value'].iloc[0]
    np.testing.assert_allclose(actual, reference, rtol=1e-10)

In [ ]:
small_treatment = [0, 0, 0, 10, 20]
small_control = [0, 0, 0, 0, 10]

point_estimate = np.mean(small_treatment) - np.mean(small_control)

bootstrap_ci = bootstrap_difference(
    small_treatment,
    small_control,
    reps=5000,
    seed=20260917
)

point_estimate, bootstrap_ci

In [ ]:
effects[effects["metric"] == "spend"][
    ["arm", "effect", "ci_low", "ci_high", "bootstrap_low", "bootstrap_high"]
]

### 多重比较

`p_holm` 用于两项收入检验。`ci_low/ci_high` 是单项区间，`family_ci_*` 为更保守的 Bonferroni 家族区间，不应混淆。辅助指标仅用于解释。

In [ ]:
display(effects[effects.metric != 'spend'][['arm','metric','mean_control','mean_treatment','effect','ci_low','ci_high','p_value']])
display(Image(filename=str(ROOT / 'reports/figures/revenue_effects.png')))

In [ ]:
effects[effects["metric"] == "spend"][
    ["arm", "p_value", "p_holm"]
].sort_values("p_value")

## 6. 探索性人群分析

先定义三个实验前维度，再以 HC3 联合 Wald 交互检验探索效果差异。交互检验与具体人群有效性是不同问题，不能用当前样本同时筛选和验证策略。

In [ ]:
interactions, subgroups = heterogeneity(df, config['alpha'])
display(interactions)
display(subgroups[['dimension','subgroup','arm','n_treatment','buyers_treatment','n_control','buyers_control','effect','ci_low','ci_high']])

In [ ]:
# 按“客户类型 × 邮件策略”计算平均收入
group_means = (
    df.groupby(["customer_type", "segment"], observed=True)["spend"]
      .mean()
      .unstack("segment")
)

# 在同一种客户类型内部，与不发邮件组比较
group_means["男装邮件增量"] = (
    group_means["Mens E-Mail"] - group_means["No E-Mail"]
)

group_means["女装邮件增量"] = (
    group_means["Womens E-Mail"] - group_means["No E-Mail"]
)

display(group_means)

判断男（女）装邮件对新客户和已有客户的效果是否不同？（近12月新客户男（女）装邮件增量-已有客户男（女）装邮件增量）

In [ ]:
effect_difference = (
    group_means.loc["近12月新客户", ["男装邮件增量", "女装邮件增量"]]
    - group_means.loc["已有客户", ["男装邮件增量", "女装邮件增量"]]
)

display(effect_difference)

## 7. 成本情景

这些毛利率与触达成本是假设，数据不含实际利润。净收益下界基于 Bonferroni 区间，仅包含收入估计的不确定性。

In [ ]:
scenarios = profit_scenarios(effects, config)
display(scenarios.head(15))
scenarios.groupby(['arm','scenario_signal']).size()

In [ ]:
# 根据收入效果、假设毛利率和触达成本，生成所有情景
scenarios = profit_scenarios(effects, config)

# 只看“女装邮件、毛利率40%”的情景
women_40 = scenarios.loc[
    (scenarios["arm"] == "Womens E-Mail")
    & (scenarios["assumed_margin"] == 0.4),
    [
        "assumed_contact_cost_usd",
        "net_per_customer",
        "net_ci_low_family",
        "net_ci_high_family",
        "scenario_signal",
    ],
]

display(women_40)

## 8. 下一轮样本量

采用历史最大组内标准差作为规划输入，另做标准差放大情景。MDE 为业务值得识别的绝对收入差，功效针对单个预设比较。不是事后功效。

In [ ]:
planning = sample_size_plan(metrics, config)
planning

## 9. 结果追溯与决策

同一配置和随机种子应复现效果表；如你改了配置，请先重新运行分析入口，再运行笔记本。

In [ ]:
saved = pd.read_csv(ROOT / 'reports/tables/effects.csv')
keys = ['arm','metric']
left = effects.set_index(keys).sort_index()
right = saved.set_index(keys).sort_index()
columns = ['effect','p_value','p_holm','ci_low','ci_high','bootstrap_low','bootstrap_high','family_ci_low','family_ci_high']
np.testing.assert_allclose(left[columns], right[columns], rtol=1e-10, atol=1e-12, equal_nan=True)
print('重算结果与报告效果表一致。')
display(Markdown((ROOT / 'reports/decision_summary.md').read_text(encoding='utf-8')))

## 10. 自测

1. 用一句话说清这项实验估计的是什么。
2. 如果转化率上升但收入区间跨零，你会怎么建议？
3. 为什么不能从两项检验的显著性差异判断两种邮件的优劣？
4. 如何把探索性人群结论转成下一轮可验证假设？
5. 哪些信息缺失阻止你声称该方案已盈利？